# mini-beatrix-3 — the v3 routine (steering-and-arms craft)

**Craft**: `mini-beatrix-3` — d1024 × L`DEPTH` (24 or 28), ctx 4096, a governed CausalSplatHUB in **every** block
(4 books × 64 anchors @ D128, supply 0.5×D), 3 banks @ ff1024, head 256@256. ~300–350M params.
Plan of record: claude-mind `history/plans/2026-09-15_beatrix_v3_plan_v2.md` (S1–S14, register A1–E, routine §4) ·
recipe `history/plans/2026-09-12_v3_recipe_v0.md` (terms A–P) · pre-flight `history/dockets/2026-09-05_bytelex_plan/v3_preflight_needs_2026-09-19.md`.

**What this notebook is**: the v2 mission notebook re-aligned to the v3 plan. It runs NOTHING on the mission craft until
the decision block in C1 is filled — every decision the record leaves to the program lead is a named placeholder that
refuses (`None`) rather than a default that decides for him. The toy switch (`LOCAL = True`) runs the whole notebook on a
CPU-sized craft with labelled TEST values so the plumbing is exercised end to end.

**Laws embodied** (all on record): supply K ≤ 2·D per book · governor from birth (min-sep, post-step, identity when slack) ·
Muon + pure Adam wd 0, never AdamW · flat LR after a 200-step warmup (the anneal's multiplier is the lead's) · bf16-train /
fp8-ship / never fp16 · eager on Blackwell (compile retired by measurement 2026-08-26) · crash-safe resume-first ·
per-boundary report + checkpoint + arm anchors (ship-complete) · the epoch cap on every finite corpus computed BEFORE a
mix ships · every stage arm quiet (per-member abstention on off-domain rows) · a faulty arm is masked out, never a crash ·
a CERTIFIED red flag halts cleanly (archive + record + refusal to auto-resume); uncertified flags are watched, never halts.

**Price** (canon L7, 4090-measured, RTX PRO 6000 = ESTIMATE until the Colab bench cell runs): 64.4B bytes at eager speed
≈ 13.7 days at L24, 16.4 days at L28. The 2026-08-26 ruling stands beside it ("26 days isn't reasonable").


In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART RUNTIME if a version line below changes.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
%pip install -q "datasets>=5.0.0" "geolip-alephllm @ git+https://github.com/AbstractEyes/alephllm@ALEPHLLM_COMMIT_PIN"
%pip install -q "amoe-lora @ git+https://github.com/AbstractEyes/amoe-lora@e21baea" --no-deps
%pip install -q "geolip-bytelex @ git+https://github.com/AbstractEyes/geolip-bytelex@e8fc82d" --no-deps
import geolip.alephllm as A
print('geolip.alephllm', A.__version__)
assert tuple(int(x) for x in A.__version__.split('.')) >= (0, 8, 9), (
    'RESTART REQUIRED: runtime still holds the old package — Runtime > Restart, then rerun from C1')
try:
    import amoe; print('amoe', amoe.__version__)
except ImportError:
    print('amoe-lora NOT importable — stage arms (STAGE_ARMS) cannot attach until it is')
try:
    import geolip.bytelex; print('geolip.bytelex importable (the atlas coverage audit instrument)')
except ImportError:
    print('geolip-bytelex NOT importable — the coverage audit needs it')

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')

# ============================ THE DECISION BLOCK ============================
# Each name below is a decision the record leaves to the program lead. None
# REFUSES: the cell that needs it stops with the record beside the name. Fill
# the value, rerun. Nothing here is a default that decides for him.
DEPTH = None            # 24 | 28 blocks. Price (canon L7, ESTIMATE on the RTX PRO 6000): 13.7 d | 16.4 d for 64.4B.
DATA_SCALE = 4.0        # x the 2s schedule, RAW BYTES (S14a; the lead's direction 2026-09-16 19:35, filed). Fused-unit accounting = S14b (open).
EPOCH_CAP = None        # epochs per finite corpus per stage: 2.0 (the law text, curriculum.py) | 4.0 (the audit threshold MAX_EPOCHS).
REBALANCE_TO = None     # where a capped corpus's weight goes: 'natural' | 'generators' (the record's precedent, <=35% each, then natural) | 'hold' (stages at 1x, the extra to general text).
ANNEAL_LR_SCALE = None  # the anneal's lower-LR multiplier (recipe term H, 0 seeds; the v2 anneal ran at 1.0 = a diet change, not an LR decay).
ARM_SPEC = None         # 'LAWFUL_16x8' (K = 2D per the supply law; 0 seeds) | 'WIDE_1024' (certified 2 seeds on the ladder; 8x over the law, a flagged exception).
STAGE_ARMS = []         # rows (name, phase, lam, seed): e.g. ('s3_rules', 'curriculum_s3', 1.0, 0). EMPTY = no arm attaches. lam 1.0 = template/capability constant; 2.0 = hard pools (S-1903).
FUSION = None           # weak-token fusion at the input plane (S14b): a config, or the literal 'WAIVED'.
COVERAGE_AUDIT = None   # the atlas coverage audit over the pretraining mix + minted programs (routine step 1; D2; S9): a report path, or 'WAIVED'.
GUARD_WAIVER = None     # read ONLY if the L6 ledger certifies no guard: 'WAIVED' runs with every guard in watch mode.
HEAD_BIRTH = None       # the born-with-function doctrine (S-1004): 'revival_birth' (the closed-form solve at step 0, address frozen after; C7 = its screen) | 'born_null' (the 2s birth form; buried 3/3 — a flagged exception) | 'WAIVED'.
HUB_GAIN = None         # training under the hub K-alpha gain (constants of record: gamma .5, rho 1/64 at u_ref, bounds [.5, 2]); NOT in the library — None = off.
THINK = DIVERSITY = CONDENSING = TOOL_ARMS = None   # post-training programs (S13 / S6 / S7): placeholders, see the last cells.
QUIET_TRUNK_GRAD = False   # a CERTIFICATION item, not the lead's: False = the certified semantics (only the arms feel the abstention term).
RESUME_AFTER_HALT = False  # per session: a run halted by a certified guard refuses to continue until this is True (the archive is read first).
LOCAL = False           # True = the toy path: a CPU-sized craft, synthetic phases, every decision above filled with a labelled TEST value.
# ===========================================================================
LOCAL = LOCAL or os.environ.get('ALEPHLLM_NB_LOCAL') == '1'
CRAFT = 'mini-beatrix-3'
if LOCAL:
    CRAFT = 'v3-local-toy'
    TEST = dict(DEPTH=2, EPOCH_CAP=4.0, REBALANCE_TO='natural', ANNEAL_LR_SCALE=0.5, ARM_SPEC='LAWFUL_16x8',
                STAGE_ARMS=[('s0_toy', 'curriculum_s0', 1.0, 0), ('s1_toy', 'curriculum_s1', 1.0, 1)],
                FUSION='WAIVED', COVERAGE_AUDIT='WAIVED', GUARD_WAIVER='WAIVED', HEAD_BIRTH='born_null')
    globals().update(TEST)
    print('LOCAL TOY PATH: the decision block is filled with TEST values', TEST)
    print('  (these never reach the mission craft; the mission craft refuses on None)')

_RECORD = {
    'DEPTH': 'canon L7 (S-1603); sizing note 2026-09-15', 'EPOCH_CAP': 'curriculum.py epoch-cap law (~2) vs audit MAX_EPOCHS 4',
    'REBALANCE_TO': 'the 2026-08-15 rebalance precedent (generators first, then natural)', 'ANNEAL_LR_SCALE': 'recipe term H; CONTROL BOUNDARY 12',
    'ARM_SPEC': 'S-801 supply law vs the certified WIDE-1024 ladder geometry (recipe B)', 'FUSION': 'plan S14b',
    'COVERAGE_AUDIT': 'plan routine step 1 / D2 / S9 (instrument at arm scale only)', 'GUARD_WAIVER': 'L6 certification rule (falsifier)',
    'HEAD_BIRTH': 'S-1004 born-with-function + global-frame guarantee (MANIFEST 09-01 amendments)'}

def require(*names):
    missing = [n for n in names if globals().get(n) is None]
    if missing:
        for n in missing:
            print(f'  DECISION MISSING: {n:<16} record: {_RECORD.get(n, "plan v2")}')
        raise SystemExit(f'refusing: fill {missing} in the C1 decision block (LOCAL=True for the toy path)')

HF_TOKEN = _token()
OUT_DIR = os.environ.get('ALEPHLLM_NB_OUT', './alephllm_runs')
import torch
DEVICE = 'cuda' if torch.cuda.is_available() and os.environ.get('ALEPHLLM_NB_CPU') != '1' else 'cpu'
print(f'craft {CRAFT} · device {DEVICE} · token {"present" if HF_TOKEN else "ABSENT (local artifacts only; Cell T stays disarmed for the mission)"}')


In [ ]:
# C2 (P) — PREFLIGHT: build the preset for the chosen depth, law checks, bit-exact births,
# the micro-batch LADDER bench at a fixed 262,144 tokens/step, the price line. No training.
import time, math, torch
from geolip.alephllm.presets import make_v3_preset, PRESETS, AlephLMConfig, _copy_train
from geolip.alephllm.model.alephlm import AlephLM
from geolip.alephllm.model.governor import govern_model
require('DEPTH', 'EPOCH_CAP', 'REBALANCE_TO', 'ANNEAL_LR_SCALE')

p = make_v3_preset(DEPTH, data_scale=DATA_SCALE, epoch_cap=EPOCH_CAP, rebalance_to=REBALANCE_TO, name=CRAFT)
p.train.phase_lr_scale = {'anneal': float(ANNEAL_LR_SCALE)}   # both anneal phases; 1.0 = the flat form verbatim
if HEAD_BIRTH == 'revival_birth':
    p.train.head_addr_frozen = True   # the doctrine: the address is solved at birth, then frozen (W_s trains)
if LOCAL:   # the toy craft: same laws, CPU size, synthetic phases
    p.model = AlephLMConfig(name=CRAFT, d_model=64, n_layers=DEPTH, n_heads=1, context=512,
                            hub_layers=tuple(range(DEPTH)), hub_K=4, hub_D=8, hub_const=2,   # supply 0.5xD like the mission
                            bank_experts=3, bank_ff=64, head_K=8, head_D=16, hub_chunk=32, hub_ckpt=0)
    p.train.micro_batch, p.train.grad_accum, p.train.warmup_steps = 2, 1, 2
    p.train.log_every, p.train.health_every, p.train.eval_every = 2, 4, 8
    p.train.ckpt_every, p.train.tb_upload_every, p.train.val_tokens, p.train.canary_episodes = 8, 1000, 256, 4
    p.curriculum = [dict(ph, dataset='synthetic', planned_tokens=4096) for ph in p.curriculum]
PRESETS[CRAFT] = p
ctrl = AlephLMConfig.from_dict(p.model.to_dict()); ctrl.name = CRAFT + '-control'; ctrl.hub_layers = ()
PRESETS[CRAFT + '-control'] = type(p)(model=ctrl, train=_copy_train(p.train), curriculum=[dict(x) for x in p.curriculum],
                                     data_scale=p.data_scale, epoch_cap=p.epoch_cap, rebalance_to=p.rebalance_to)
cfg, tc = p.model, p.train
PLANNED = sum(ph['planned_tokens'] for ph in p.curriculum)
print(f"{cfg.name}: depth {cfg.n_layers} · d {cfg.d_model} · ctx {cfg.context} · hubs {len(cfg.hub_layers)}/{cfg.n_layers} × "
      f"{cfg.hub_const} books × {cfg.hub_K}@D{cfg.hub_D} (supply {cfg.hub_K/cfg.hub_D:.2f}×D) · planned {PLANNED/1e9:.2f}B bytes")
for ph in p.curriculum:
    print(f"   {ph['name']:<18} {ph['dataset']:<16} {ph['planned_tokens']/1e9:8.3f}B")
assert cfg.hub_K <= 2 * cfg.hub_D, 'supply law violated — this preset should not exist'
assert tc.compile is False, 'eager on Blackwell (the 2026-08-26 law)'
torch.manual_seed(tc.seed)
model = AlephLM(cfg).to(DEVICE)
n = model.param_count(); print(f'{n/1e6:.1f}M params')
hits = govern_model(model, tc.governor_theta)
print(f'governor birth check: {hits} hits (must be 0 at mission D)' + (' — toy D, not gated' if LOCAL else ''))
assert hits == 0 or LOCAL
x = torch.randint(0, 255, (2, min(cfg.context, 256)), device=DEVICE)
with torch.no_grad():   # C6 null paths: bank + head aleph contribute exactly zero at init
    a = model(x, disable_bank=True, disable_head_aleph=True).logits
    b = model(x).logits
    assert torch.equal(a, b), 'C6 violated — born-null paths are not null'
print('C6 null paths bit-exact at birth')

# the ladder: micro-batch by fit at a FIXED 262,144 tokens/step (16 x 4096); the first
# rung that fits is the session's micro_batch, grad_accum follows. An arm rides the
# bench when STAGE_ARMS is non-empty (the arm's forward is part of the step it prices).
TOK_STEP = tc.micro_batch * tc.grad_accum * cfg.context
if DEVICE == 'cuda':
    import platform
    frac = 0.73 if platform.system() == 'Windows' else 0.98
    if platform.system() == 'Windows':
        print('WINDOWS (WDDM): fraction 0.73 — the ~18 GB local cap; the mission card is Linux at 0.98')
    torch.cuda.set_per_process_memory_fraction(frac)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
    abort_gb = 88 if total_gb > 90 else 0.73 * total_gb
    if STAGE_ARMS:
        from geolip.alephllm.train.arms import StageArm, ArmProgramConfig, StageArmProgram, LAWFUL_16x8, WIDE_1024
        spec = {'LAWFUL_16x8': LAWFUL_16x8, 'WIDE_1024': WIDE_1024}[ARM_SPEC]
        class _T: pass
        _t = _T(); _t.raw_model = model; _t.device = DEVICE; _t._trunk_params = list(model.parameters())
        prog = StageArmProgram(ArmProgramConfig(arms=[StageArm('bench', 'bench', spec=dict(spec))])).bind(_t)
        prog.attach(prog.by_name['bench']); print('bench with one arm attached', spec)
    model.train(); best = None
    for mb in (16, 8, 4, 2, 1):
        accum = max(1, TOK_STEP // (mb * cfg.context))
        try:
            opt_probe = torch.optim.SGD([q for q in model.parameters() if q.requires_grad], lr=0.0)
            xb = torch.randint(0, 255, (mb, cfg.context), device=DEVICE)
            torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize(); t0 = time.time()
            for i in range(3):
                with torch.autocast('cuda', dtype=torch.bfloat16):
                    out = model(xb, targets=xb)
                out.loss.backward(); opt_probe.zero_grad(set_to_none=True)
            torch.cuda.synchronize(); sps = (time.time() - t0) / 3
            vram = torch.cuda.max_memory_allocated() / 2**30
            tps = mb * cfg.context / sps
            print(f'mb {mb:2d} x accum {accum:2d}: {sps:.2f}s/micro · {tps/1e3:.1f}k tok/s · peak {vram:.1f} GB' + (' OVER BUDGET' if vram > abort_gb else ''))
            if vram <= abort_gb and best is None:
                best = (mb, accum, tps)
            del out, xb, opt_probe; torch.cuda.empty_cache()
        except torch.cuda.OutOfMemoryError:
            print(f'mb {mb:2d}: OOM'); torch.cuda.empty_cache()
    assert best is not None, 'no rung fits — the card cannot carry this depth'
    tc.micro_batch, tc.grad_accum, tps = best[0], best[1], best[2]
    days = PLANNED / tps / 86400
    card = torch.cuda.get_device_name(0)
    label = 'MEASURED on the mission card' if total_gb > 90 else f'ESTIMATE (measured on {card}, not the mission card)'
    print(f'PRICE: {PLANNED/1e9:.1f}B bytes at {tps/1e3:.1f}k tok/s = {days:.1f} days — {label}; the decision is the lead\'s (08-26: "26 days isn\'t reasonable")')
else:
    print('CPU: the ladder bench and the price line need a card (skipped)')
del model; torch.cuda.empty_cache() if DEVICE == 'cuda' else None
print('PREFLIGHT PASS — the data cell, the guard cell and the push cell must pass before Cell T')


In [ ]:
# CARD-ONLY
# C2b — WHERE DOES THE TIME GO. Run after a restart, before committing to Cell T (needs the card).
# Splits one training step into: data / forward / backward / optimizer+governor.
import time, torch
from geolip.alephllm import get_preset
from geolip.alephllm.model.alephlm import AlephLM
from geolip.alephllm.model.governor import govern_model
from geolip.alephllm.train.optim import build_optimizers
from geolip.alephllm.data.streams import build_stream
from geolip.alephllm.data.tokenizers import build_tokenizer
import geolip.alephllm as A
print('package', A.__version__); assert tuple(int(x) for x in A.__version__.split('.')) >= (0,8,9), 'STALE RUNTIME'
p = get_preset(CRAFT); cfg, tc = p.model, p.train
torch.manual_seed(0)
m = AlephLM(cfg).cuda(); m.train()
opts = build_optimizers(m, tc.muon_lr, tc.muon_momentum, tc.adam_lr)
def sync(): torch.cuda.synchronize()
x = torch.randint(0, 255, (tc.micro_batch, cfg.context), device='cuda')
# warmup (autotune, lazy inits)
with torch.autocast('cuda', dtype=torch.bfloat16):
    m(x, targets=x).loss.backward()
m.zero_grad(set_to_none=True); sync()
# (a) GPU compute, synthetic data: one full accumulation step
t0 = time.time(); tf = tb = 0.0
for _ in range(tc.grad_accum):
    s = time.time()
    with torch.autocast('cuda', dtype=torch.bfloat16):
        out = m(x, targets=x)
    sync(); tf += time.time() - s; s = time.time()
    (out.loss / tc.grad_accum).backward(); sync(); tb += time.time() - s
s = time.time()
torch.nn.utils.clip_grad_norm_(m.parameters(), tc.grad_clip)
for o in opts: o.step()
sync(); topt = time.time() - s
s = time.time(); hits = govern_model(m, tc.governor_theta); sync(); tgov = time.time() - s
m.zero_grad(set_to_none=True)
gpu_step = time.time() - t0
print(f'GPU step (synthetic data): {gpu_step:.1f}s = fwd {tf:.1f} + bwd {tb:.1f} + opt {topt:.1f} + governor {tgov:.2f} (hits {hits})')
print(f'  -> pure-compute ceiling: {tc.micro_batch*cfg.context*tc.grad_accum/gpu_step/1e3:.0f}k tok/s')
# (b) the data pipeline alone: how fast can the stream feed 2M tokens?
tok = build_tokenizer(cfg.tokenizer)
st = build_stream('wikitext-103', tok, cfg.context, tc.micro_batch, seed=1, role='train')
st.next_batch()  # open/download outside the clock
n_tok = 0; s = time.time()
while n_tok < 2_000_000:
    b = st.next_batch(); n_tok += b.numel()
dt = time.time() - s
print(f'stream: {n_tok/dt/1e3:.0f}k tok/s CPU -> {524288/(n_tok/dt):.1f}s of data per step if serial')
print('VERDICT: whichever line dominates the observed s/step is the bottleneck.')


In [ ]:
# CARD-ONLY
# C2c — THE COMPILE GATE (the gate the law names, kept as a gate). Compile is RETIRED
# on Blackwell BY MEASUREMENT (2026-08-26: NaN to the embeddings on every graph; the
# 4090 reproduced it 2026-09-19 with 488/491 non-finite gradients at 1.45x speed).
# It is not a utilization lever. Law: compile trains only if the GRAD-VECTOR parity
# below passes on the training hardware — never loss-only (loss-only gating shipped
# a NaN once). Expected result: GATE FAIL; eager stands.
import time, torch
from geolip.alephllm import get_preset
from geolip.alephllm.model.alephlm import AlephLM
p = get_preset(CRAFT); cfg, tc = p.model, p.train
torch.manual_seed(0)
m = AlephLM(cfg).cuda(); m.train()
x = torch.randint(0, 255, (tc.micro_batch, cfg.context), device='cuda')
def grads(model):
    with torch.autocast('cuda', dtype=torch.bfloat16):
        model(x, targets=x).loss.backward()
    g = {n: p.grad.detach().clone() for n, p in model.named_parameters() if p.grad is not None}
    model.zero_grad(set_to_none=True)
    return g
g_eager = grads(m)
mc = torch.compile(m)
with torch.autocast('cuda', dtype=torch.bfloat16):
    mc(x, targets=x).loss.backward()   # compile warmup (slow once)
m.zero_grad(set_to_none=True)
g_comp = {}
with torch.autocast('cuda', dtype=torch.bfloat16):
    mc(x, targets=x).loss.backward()
for n, prm in m.named_parameters():
    if prm.grad is not None: g_comp[n] = prm.grad.detach().clone()
m.zero_grad(set_to_none=True)
bad = [(n, float((g_eager[n] - g_comp[n]).abs().max())) for n in g_eager
       if not torch.isfinite(g_comp[n]).all() or
          float((g_eager[n] - g_comp[n]).abs().max()) > 1e-2 * (1e-6 + float(g_eager[n].abs().max()))]
nonfinite = [n for n in g_comp if not torch.isfinite(g_comp[n]).all()]
print('non-finite compiled grads:', nonfinite[:5] if nonfinite else 'NONE')
print('parity failures (rel>1e-2):', bad[:5] if bad else 'NONE')
torch.cuda.synchronize(); t0 = time.time()
for _ in range(3):
    with torch.autocast('cuda', dtype=torch.bfloat16):
        mc(x, targets=x).loss.backward()
    m.zero_grad(set_to_none=True)
torch.cuda.synchronize()
sps = (time.time() - t0) / 3
print(f'compiled micro fwd+bwd: {sps:.2f}s -> projected step ~{sps*tc.grad_accum:.0f}s '
      f'({tc.micro_batch*cfg.context*tc.grad_accum/(sps*tc.grad_accum)/1e3:.0f}k tok/s)')
if not nonfinite and not bad:
    print('GATE PASS — set TrainConfig.compile=True for the session (p.train.compile = True before prepare)')
else:
    print('GATE FAIL — eager stands; report the lines above')


In [ ]:
# CARD-ONLY
# C2d — INSIDE ONE HUB: per-op forward timing on the real card. Names the slow op.
import time, torch, torch.nn.functional as F
from geolip.alephllm import get_preset
from geolip.alephllm.model.attention import CausalSplatHUB
from geolip.alephllm.model.bank import AnchoredBank
p = get_preset(CRAFT); cfg, tc = p.model, p.train
torch.manual_seed(0)
hub = CausalSplatHUB(cfg.d_model, cfg.hub_K, cfg.hub_D, cfg.tau,
                     chunk=cfg.hub_chunk, n_const=cfg.hub_const).cuda()
bank = AnchoredBank(cfg.d_model, cfg.bank_experts, cfg.bank_ff, cfg.tau, cfg.gate_init).cuda()
B, n, d = tc.micro_batch, cfg.context, cfg.d_model
x = torch.randn(B, n, d, device='cuda')
def t(fn, reps=3):
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
        fn(); torch.cuda.synchronize()
        s = time.time()
        for _ in range(reps): fn()
        torch.cuda.synchronize()
    return (time.time() - s) / reps
with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
    qc, kc = hub._code_cat_qk(x); v = hub.v(x)
    C = min(hub.chunk, n); pad = (-n) % C; nc = (n + pad) // C
    vc = (F.pad(v, (0,0,0,pad)) if pad else v).view(B, nc, C, d)
    mask = hub._mask(C, x.device, vc.dtype)
    K2 = qc.shape[-1]
    qv = (F.pad(qc, (0,0,0,pad)) if pad else qc).view(B, nc, C, K2)
    kv = (F.pad(kc, (0,0,0,pad)) if pad else kc).view(B, nc, C, K2)
    S = torch.einsum('bick,bicd->bikd', kv, vc)
    L = hub._prefix(nc, qc.device, qc.dtype)
    P = torch.matmul(L, S.reshape(B, nc, -1)).view_as(S)
    att = torch.einsum('bick,bijk->bicj', qv, kv) * mask
rows = [
 ('code_cat_qk (1 softmax)', t(lambda: hub._code_cat_qk(x))),
 ('v proj                 ', t(lambda: hub.v(x))),
 ('S einsum (write)       ', t(lambda: torch.einsum('bick,bicd->bikd', kv, vc))),
 ('prefix matmul P        ', t(lambda: torch.matmul(L, S.reshape(B, nc, -1)))),
 ('att within-chunk       ', t(lambda: torch.einsum('bick,bijk->bicj', qv, kv) * mask)),
 ('num read (qc.P + att.v)', t(lambda: torch.einsum('bick,bikd->bicd', qv, P) + att @ vc)),
 ('FULL hub.forward       ', t(lambda: hub(x))),
 ('bank.forward           ', t(lambda: bank(x))),
]
for name, dt in rows: print(f'{name} {dt*1e3:8.1f} ms')
d_ = dict(rows)
print(f"-> {cfg.n_layers} layers x (hub+bank) = {(d_['FULL hub.forward       '] + d_['bank.forward           '])*cfg.n_layers:.1f}s per micro-forward")
print(f'-> dtypes: qc {qc.dtype}, v {v.dtype}  (bf16 expected everywhere now)')

In [ ]:
# CARD-ONLY
# C2e — hub_chunk sweep on the real card: S/P traffic ~ 1/C, att FLOPs ~ C.
# 4090 evidence (2026-08-26): flat 256–1024 wall-clock, peak memory favors larger.
# The Blackwell rules; if a non-256 chunk wins >10%%, set p.model.hub_chunk before
# build (config field, no arch change, checkpoint-compatible).
import time, torch
from geolip.alephllm import get_preset
from geolip.alephllm.model.attention import CausalSplatHUB
p = get_preset(CRAFT); cfg, tc = p.model, p.train
torch.manual_seed(0)
x = torch.randn(tc.micro_batch, cfg.context, cfg.d_model, device='cuda')
for C in (256, 512, 768, 1024, 1536):
    hub = CausalSplatHUB(cfg.d_model, cfg.hub_K, cfg.hub_D, cfg.tau,
                         chunk=C, n_const=cfg.hub_const).cuda()
    def run():
        with torch.autocast('cuda', dtype=torch.bfloat16):
            loss = hub(x).square().mean()
        loss.backward(); hub.zero_grad(set_to_none=True)
    run(); torch.cuda.synchronize()
    s = time.time()
    for _ in range(3): run()
    torch.cuda.synchronize()
    print(f'chunk {C:5d}: {(time.time()-s)/3*1e3:7.1f} ms fwd+bwd/layer  '
          f'peak {torch.cuda.max_memory_allocated()/2**30:.1f} GB')
    torch.cuda.reset_peak_memory_stats()
    del hub; torch.cuda.empty_cache()

In [ ]:
# CARD-ONLY
# C2f — THE RESIDUAL: full-model micro fwd+bwd vs the hub+bank sum, plus a
# micro_batch=8 fit/speed probe. C2d/C2e see only hub+bank; the step arithmetic
# leaves a remainder unaccounted (LayerNorms x2 per block, embed, head, loss, trainer
# overhead). This names it — and if micro 8 fits, per-micro fixed costs halve.
import time, torch
from geolip.alephllm import get_preset
from geolip.alephllm.model.alephlm import AlephLM
p = get_preset(CRAFT); cfg, tc = p.model, p.train
torch.manual_seed(0)
model = AlephLM(cfg).cuda()
model.train()

def micro(B):
    idx = torch.randint(0, 256, (B, cfg.context), device='cuda')
    tgt = torch.randint(0, 256, (B, cfg.context), device='cuda')
    def run():
        with torch.autocast('cuda', dtype=torch.bfloat16):
            out = model(idx, targets=tgt)
        out.loss.backward()
        model.zero_grad(set_to_none=True)
    try:
        run(); torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        s = time.time()
        for _ in range(2): run()
        torch.cuda.synchronize()
        ms = (time.time()-s)/2*1e3
        pk = torch.cuda.max_memory_allocated()/2**30
        print(f'micro_batch {B}: {ms/1e3:6.2f} s fwd+bwd  peak {pk:.1f} GB  '
              f'-> step ~{ms/1e3 * (64//B):5.1f} s at {64*cfg.context/1e3:.0f}k tok (x{64//B} micros)')
        return ms
    except torch.cuda.OutOfMemoryError:
        print(f'micro_batch {B}: OOM'); torch.cuda.empty_cache(); return None

m4 = micro(tc.micro_batch)
# hub+bank accounting at this chunk (from C2e's per-layer fwd+bwd + one
# recompute-fwd under ckpt) vs the measured full micro:
if m4:
    print(f'-> subtract (C2e ms + C2d hub-fwd ms) x {cfg.n_layers} from the micro above:')
    print('   the remainder IS the residual (LN/embed/head/loss/overhead).')
micro(8)   # fits? halves per-micro fixed costs at identical tokens/step


In [ ]:
# C-DATA — THE DATA PLANE at DATA_SCALE: stage budgets, every finite corpus's epochs
# per stage BEFORE the mix ships (the epoch-cap law), the rebalance under the chosen
# rule, the cross-stage epoch column, and the coverage-audit gate. Pure computation.
from geolip.alephllm.data import curriculum as C
sc = None
try:
    require('EPOCH_CAP', 'REBALANCE_TO')
    sc = C.scaled_curriculum(DATA_SCALE, EPOCH_CAP, REBALANCE_TO)
except SystemExit as e:
    print(e)
    print('\nTHE TABLES FOR EVERY RULE (so the lead can rule; nothing is applied):')
    for cap in (2.0, 4.0):
        for rule in C.REBALANCE_RULES:
            s = C.scaled_curriculum(DATA_SCALE, cap, rule)
            moved = [r for r in s['table'] if r[2] != r[3]]
            hot = {n: e for n, e in s['cross_stage'].items() if e > cap}
            print(f'  cap {cap:g} / {rule:<10}: {len(moved):2d} components move; held {s["held_tokens"]/1e9:.1f}B; '
                  f'cross-stage over the cap: {hot or "none"}')
    raise SystemExit('refusing: EPOCH_CAP and REBALANCE_TO are the lead\'s')
print(f'scale x{DATA_SCALE:g} · cap {sc["epoch_cap"]:g} epochs · rule {REBALANCE_TO} · stages '
      f'{sum(sc["stage_tokens"].values())/1e9:.2f}B' + (f' · held {sc["held_tokens"]/1e9:.1f}B -> fineweb_main' if sc['held_tokens'] else ''))
if sc['flags']:
    print('FLAGS', sc['flags'])
print(f"{'stage':<14}{'component':<20}{'w before':>9}{'w after':>9}{'ep before':>10}{'ep after':>9}")
for st, n, wb, wa, eb, ea in sc['table']:
    mark = '  <- moved' if wb != wa else ''
    print(f'{st:<14}{n:<20}{wb:>9.3f}{wa:>9.3f}{eb:>10.2f}{ea:>9.2f}{mark}')
print('cross-stage epochs (total over the curriculum):', sc['cross_stage'])
for st, mix in sc['mixes'].items():
    ballast = sum(w for n, w in mix if n in C.NATURAL)
    top = max([w for n, w in mix if n.endswith('-synth')], default=0.0)
    assert ballast >= C.MIN_BALLAST - 1e-9, (st, ballast, 'ballast law')
    assert top <= C.MAX_GENERATOR_SHARE + 1e-9, (st, top, 'generator share law')
print('ballast >= 30% and every generator <= 35% in every stage: OK')
print('NOTE: the infinite recipients (cosmo-young, fineweb-good, wikipedia-en, gutenberg) are infinite by the '
      'registry; their supply at the 4x demand is OWED a measurement (the coverage audit).')
require('COVERAGE_AUDIT')
if COVERAGE_AUDIT == 'WAIVED':
    print('COVERAGE AUDIT WAIVED by the lead (routine step 1 / D2 / S9): the curriculum phases open without it — recorded')
else:
    print('coverage audit report:', COVERAGE_AUDIT, '(the atlas calls: WeightField.caveat / coverage set-ops / '
          'mint_lexicon_program over N sampled rows per source — an instrument validated at ARM scale only)')
DATA_PLANE = C.data_plane(DATA_SCALE, EPOCH_CAP, REBALANCE_TO)
print('data plane fingerprint (asserted on every resume):', DATA_PLANE)


In [ ]:
# C-GUARD — THE RED-FLAG CORE from the shipped L6 certification ledger. A guard halts
# ONLY with a certification on record (fires on its own induced fault within 600 steps
# AND silent on both healthy seeds); everything else is watched. The library's census
# flags fire on healthy births — recorded, never interrupts. G4-G6 are boundary reads.
import json, os
from huggingface_hub import hf_hub_download
from geolip.alephllm.train.guards import GuardConfig, certification_from_ledger, GUARDS
from geolip.alephllm.presets import TRAINING_REPO
LEDGER_PATH = 'v3_preflight/l6_guards/v3_l6_guards_ledger.json'
led = None
for cand in ([os.environ.get('ALEPHLLM_L6_LEDGER')] if os.environ.get('ALEPHLLM_L6_LEDGER') else []):
    if os.path.exists(cand):
        led = json.load(open(cand, encoding='utf-8')); print('ledger (local):', cand)
if led is None:
    try:
        led = json.load(open(hf_hub_download(TRAINING_REPO, LEDGER_PATH, token=HF_TOKEN), encoding='utf-8'))
        print('ledger (hub):', LEDGER_PATH)
    except Exception as e:
        print('ledger unavailable:', repr(e)[:200])
cert = certification_from_ledger(led or {})
V = (led or {}).get('verdict', {}).get('certification', {})
print(f"{'guard':<12}{'fault':>6}{'first fire':>11}{'latency':>8}{'healthy silent':>15}{'CERTIFIED':>10}")
for g, r in V.items():
    print(f"{g:<12}{str(r.get('fault_arm')):>6}{str(r.get('first_fire')):>11}{str(r.get('latency')):>8}"
          f"{str(r.get('silent_on_healthy')):>15}{str(r.get('CERTIFIED')):>10}")
if not V:
    print('NO VERDICT in the ledger (the certification run has not finished, or the ledger is missing)')
print('modes:', cert['modes'])
if not any(m == 'halt' for m in cert['modes'].values()):
    require('GUARD_WAIVER')
    print('ZERO certified guards: GUARD_WAIVER =', GUARD_WAIVER, '— every guard runs in watch mode (recorded)')
GUARD_CONFIG = GuardConfig(modes=cert['modes'], certification=cert['certification'],
                           census_dataset='synthetic' if LOCAL else 'fineweb-edu',
                           ref_window=(4, 16) if LOCAL else (1000, 2000),
                           census_every=4 if LOCAL else 100)
print('reference window (steps]:', GUARD_CONFIG.ref_window, '· norm sample every', 'log_every', '· census every', GUARD_CONFIG.census_every,
      'on one pinned batch from', GUARD_CONFIG.census_dataset, 'with every arm masked')


In [ ]:
# C-PUSH — THE PUSH PROBE (the ship-complete law): a write to the training repo under
# <craft>/v3_preflight/ and a read-back BEFORE any training. Tokenless = disarmed.
import json, time
from huggingface_hub import HfApi
from geolip.alephllm.presets import TRAINING_REPO
PUSH_OK = False
if HF_TOKEN:
    api = HfApi(token=HF_TOKEN)
    path = f'{CRAFT}/v3_preflight/push_probe.json'
    body = json.dumps({'craft': CRAFT, 'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
                       'depth': DEPTH, 'data_plane': DATA_PLANE, 'modes': GUARD_CONFIG.modes}).encode()
    api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=TRAINING_REPO, commit_message='v3 push probe')
    files = set(api.list_repo_files(TRAINING_REPO))
    PUSH_OK = path in files
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: no probe written; the mission Cell T stays DISARMED (the toy path proceeds on local artifacts)')
assert PUSH_OK or LOCAL, 'Cell T needs a verified push path'


In [ ]:
# C3 (T) — THE SESSION. Resume-first; stops at stage boundaries; every boundary ships a
# checkpoint (HubSync), the arm anchors, AND a report JSON (this cell). A CERTIFIED guard
# halt returns cleanly: the position is archived under its own name, latest.pt stays at
# the last healthy checkpoint, the manifest records the halt, and the run refuses to
# continue until RESUME_AFTER_HALT is set for a session (read the archive first).
import gc, json, os, time, torch
from geolip.alephllm import prepare
from geolip.alephllm.train import probes
from geolip.alephllm.train.instruments import model_census, toggle_ledger, special_token_gauge
from geolip.alephllm.model.governor import govern_model
require('DEPTH', 'EPOCH_CAP', 'REBALANCE_TO', 'ANNEAL_LR_SCALE', 'FUSION', 'COVERAGE_AUDIT', 'HEAD_BIRTH')
if STAGE_ARMS:
    require('ARM_SPEC')
assert PUSH_OK or LOCAL, 'the push probe did not pass'
if FUSION != 'WAIVED':
    raise SystemExit('FUSION is a config: no fused unit crosses a special or the turn-end pair, and the '
                     'raw-vs-fused accounting must be defined — not built in 0.8.9 (S14b); set FUSION="WAIVED" to run on raw bytes')
if HEAD_BIRTH not in ('revival_birth', 'born_null', 'WAIVED'):
    raise SystemExit('HEAD_BIRTH must name a mechanism of record or WAIVED')
MAX_HOURS = 10.5 if not LOCAL else 0.2

ARM_PROGRAM = None
if STAGE_ARMS:
    from geolip.alephllm.train.arms import StageArm, ArmProgramConfig, StageArmProgram, LAWFUL_16x8, WIDE_1024
    spec = {'LAWFUL_16x8': LAWFUL_16x8, 'WIDE_1024': WIDE_1024}[ARM_SPEC]
    if spec['K'] > 2 * spec['D']:
        print(f'ARM_SPEC {ARM_SPEC}: K {spec["K"]} > 2D — a flagged exception to the supply law (the certified ladder geometry)')
    ARM_PROGRAM = StageArmProgram(ArmProgramConfig(
        arms=[StageArm(name, phase, spec=dict(spec), lam=float(lam), seed=int(seed)) for name, phase, lam, seed in STAGE_ARMS],
        offdomain_dataset='synthetic' if LOCAL else 'fineweb-edu', quiet_trunk_grad=bool(QUIET_TRUNK_GRAD)))
    print('stage arms:', [(a.name, a.phase, a.lam) for a in ARM_PROGRAM.cfg.arms], '· trunk-live form: 0 seeds (the v3 rung; frozen-trunk certified 2 seeds)')

run = prepare(PRESETS[CRAFT], hf_token=HF_TOKEN, out_dir=OUT_DIR, guard=GUARD_CONFIG, arms=ARM_PROGRAM,
              resume=os.environ.get('ALEPHLLM_NB_FRESH') != '1', device=DEVICE)
if run.step == 0 and HEAD_BIRTH == 'revival_birth':
    # the born-with-function solve at step 0 (the C7 arm-B form): fit the head address in closed
    # form on a birth sample, freeze the address, log provenance — a boundary-write at birth
    from geolip.alephllm.train.revival import revive_head
    m = run.raw_model.float(); hs, bl, ys = [], [], []
    hook = m.nf.register_forward_hook(lambda mod, i, o: hs.append(o.detach().float().reshape(-1, o.shape[-1])))
    with torch.no_grad():
        for xb in run._val():
            out = m(xb[:, :-1], disable_head_aleph=True)
            bl.append(out.logits.float().reshape(-1, m.cfg.vocab_size)); ys.append(xb[:, 1:].reshape(-1))
    hook.remove()
    prov = revive_head(m, torch.cat(hs), torch.cat(bl), torch.cat(ys), theta_deg=45.0)
    m.head.proj.weight.requires_grad_(False); m.head.addr.codebook.requires_grad_(False)
    run.manifest.note(f'HEAD_BIRTH revival_birth at step 0: {json.dumps(prov, default=float)[:400]}')
    print('head solved at birth (provenance in the manifest); address frozen; W_s trains')
elif HEAD_BIRTH == 'born_null':
    run.manifest.note('HEAD_BIRTH born_null: the 2s birth form (buried 3/3 on record) — a flagged exception to S-1004')
REPORT_DIR = os.path.join(run.out_dir, 'reports', 'v3'); os.makedirs(REPORT_DIR, exist_ok=True)
_prev_anneal = {}

def boundary_report(run, tag):
    m, dev = run.raw_model, run.device
    m.eval()
    with torch.no_grad():
        pr = probes.run_all(m, run.tokenizer, dev)
        census = model_census(m, run._sample_batch())          # a VAL sample, never training rows
        ledger = toggle_ledger(m, run._val())
        arms = None
        if run.arms is not None and run.arms.attached:
            with run.arms.all_off():
                bare = toggle_ledger(m, run._val())
            ledger['bpb_arms_off'] = bare['bpb_full']; ledger['toggle_arms_off'] = bare['bpb_full'] - ledger['bpb_full']
            arms = run.arms.gauges(run._val(), None)
        sp = special_token_gauge(m, run._val())
    erank = {str(L): census['layers'][L].get('hidden_erank') for L in census['layers']}
    rep = {'tag': tag, 'step': run.step, 'tokens': run.manifest.tokens_seen, 'phase': tag,
           'next_phase': (run.manifest.current_phase() or {}).get('name'), 'lr_mult': run._phase_mult(tag),
           'probes': pr, 'census_flags': census.get('flags'), 'erank_profile': erank, 'ledger': ledger, 'special': sp,
           'governor_hits_cum': getattr(run, '_gov_hits', 0),
           'crowd_check_extra_hits': govern_model(m, run.tc.governor_theta) if run.tc.governor else None,
           'guard': run.guard.summary() if run.guard is not None else None, 'arms': arms,
           'arms_disabled': dict(run.arms.disabled) if run.arms is not None else None,
           'halt': run.manifest.halt}
    # boundary reads G5/G6 (WATCH; no interrupt of record): G5 = a member past the quiet bar
    # (+.012 fineweb) or blending (selectivity <= 1.5); G6 = termination regression across
    # anneal checkpoints (doc/reset bpb rising) — recorded with fired=False until a rule is minted
    if run.guard is not None:
        if arms:
            bad = {n: r for n, r in arms.items() if isinstance(r, dict) and (r.get('fineweb_delta', 0) > 0.012)}
            run.guard.record_boundary_read('G5', tag, {'fired': False, 'step': run.step, 'past_quiet_bar': list(bad)})
        if tag.startswith('anneal') and sp:
            prev = _prev_anneal.get('sp')
            delta = {k: sp[k] - prev[k] for k in ('doc_bpb', 'reset_bpb') if prev and k in sp and k in prev}
            run.guard.record_boundary_read('G6', tag, {'fired': False, 'step': run.step, 'delta_vs_previous': delta})
            _prev_anneal['sp'] = sp
    if run.arms is not None:
        for n in run.arms.attached:
            ck = run.arms.anchor(n, CRAFT, run.step)
            apath = os.path.join(run.out_dir, 'arms', f'{n}_step{run.step}.safetensors'); os.makedirs(os.path.dirname(apath), exist_ok=True)
            ck.save(apath); run.hub._up(apath, f'arms/{n}_step{run.step}.safetensors')
    body = json.dumps(rep, default=float)
    with open(os.path.join(REPORT_DIR, f'{tag}_step{run.step}.json'), 'w', encoding='utf-8') as f:
        f.write(body)
    run.hub.upload_bytes(body.encode(), f'reports/v3/{tag}_step{run.step}.json')
    print(probes.report(pr)); m.train()
    return rep

# 0.8.1 driver contract + the v3 halt: train(stop_at_boundary=True) RETURNS at every
# phase boundary (run._last_boundary), on a manual stop (run._interrupted — NEVER
# auto-resumed) and on a certified guard halt (run._guard_halt — NEVER auto-resumed).
t_end = time.time() + MAX_HOURS * 3600
while time.time() < t_end:
    hours_left = (t_end - time.time()) / 3600
    if hours_left < (0.2 if not LOCAL else 0.0):
        break
    run.train(max_hours=hours_left, stop_at_boundary=True, resume_after_halt=RESUME_AFTER_HALT)
    RESUME_AFTER_HALT = False       # one clearance per session, never standing
    if getattr(run, '_interrupted', False):
        print('manual stop - no auto-resume; resume state is on the hub'); break
    halt = getattr(run, '_guard_halt', None)
    if halt:
        boundary_report(run, f"guard_{halt['guard']}")
        print(f"GUARD HALT {halt['guard']} at step {halt['step']:,} in '{halt['phase']}' — archive {halt['archive']}; "
              "the session ends here; the driver (a human) decides: read the archive and the report, then a new session "
              "with RESUME_AFTER_HALT=True continues from the last healthy checkpoint (a rewind target = the last boundary archive)")
        break
    tag = getattr(run, '_last_boundary', None) or 'session_cap'
    boundary_report(run, tag)
    if run.manifest.current_phase() is None:
        print('curriculum complete — the two-phase anneal was planned from birth and has run'); break
    gc.collect(); torch.cuda.empty_cache() if DEVICE == 'cuda' else None
print('session over — resume state on the hub' if HF_TOKEN else 'session over — resume state local')


In [ ]:
# C5 — SAMPLES WITH VISIBLE SPECIALS. Run any time after prepare() (Cell T
# section); uses the in-memory model. Specials render as ⟦NAME⟧ — the eyeball
# check that the craft emits DOC at document ends (and, come anneal, END at
# turn closes). decode_visible is display-only and spoofable by content;
# the ids are the truth — never parse this string.
import torch
from geolip.alephllm.data.special_tokens import (DOC, decode_visible,
                                                 render_chat_ids)

prompt = "The history of astronomy begins with"
ids = torch.tensor(run.tokenizer.encode(prompt), device=run.device)[None]
run.raw_model.eval()
with torch.no_grad():
    out = run.raw_model.generate(ids, max_new=256, temperature=0.8)
print(decode_visible(out[0].tolist()))

# chat-frame probe (meaningful once anneal has run; before that the frame
# is out-of-distribution and the continuation shows exactly that):
chat = render_chat_ids([{"role": "user", "content": "Who are you?"}])
import numpy as np
from geolip.alephllm.data.special_tokens import MODEL as MODEL_ID
seed = np.concatenate([chat, np.array([MODEL_ID])])
ids = torch.tensor(seed, device=run.device)[None]
with torch.no_grad():
    out = run.raw_model.generate(ids, max_new=128, temperature=0.8)
print("\n--- chat probe ---")
print(decode_visible(out[0].tolist()))

In [ ]:
# C6 — THE ANNEAL WATCH. The two-phase anneal (nochat then chat) is PLANNED FROM BIRTH in
# the v3 preset (no activation step); this cell shows the multiplier in force and the
# per-checkpoint gauge table across the anneal reports (G6 = the termination read).
import json, glob, os
print('anneal LR multiplier:', run._phase_mult('anneal_nochat'), '/', run._phase_mult('anneal_mix'),
      '(1.0 = the flat form; the value is the lead\'s — recipe term H, 0 seeds)')
reps = sorted(glob.glob(os.path.join(REPORT_DIR, 'anneal*_step*.json')))
print(f"{'report':<28}{'step':>8}{'val bpb':>9}{'doc bpb':>9}{'reset bpb':>10}{'head':>8}")
for f in reps:
    r = json.load(open(f, encoding='utf-8')); led = r.get('ledger') or {}; sp = r.get('special') or {}
    v = led.get('bpb_full'); d = sp.get('doc_bpb'); rs = sp.get('reset_bpb'); h = led.get('toggle_head_aleph_off')
    print(f"{os.path.basename(f)[:27]:<28}{r['step']:>8}{(f'{v:.3f}' if isinstance(v, float) else '--'):>9}"
          f"{(f'{d:.2f}' if isinstance(d, float) else '--'):>9}{(f'{rs:.2f}' if isinstance(rs, float) else '--'):>10}"
          f"{(f'{h:+.3f}' if isinstance(h, float) else '--'):>8}")
if run.guard is not None and run.guard.boundary_reads.get('G6'):
    print('G6 termination reads:', run.guard.boundary_reads['G6'])


In [ ]:
# C4 — GROWTH TABLE from the boundary reports (reports/v3). Probe values are DICTS
# ({'acc': ...}); op reports carry no 'probes' — skipped, not crashed. Guard and arm
# columns added for v3.
import json, glob, os
from huggingface_hub import HfApi, hf_hub_download
from geolip.alephllm.presets import TRAINING_REPO
if LOCAL or not HF_TOKEN:
    files = sorted(glob.glob(os.path.join(REPORT_DIR, '*.json')))
    load = lambda f: json.load(open(f, encoding='utf-8'))
else:
    api = HfApi(token=HF_TOKEN)
    files = sorted(f for f in api.list_repo_files(TRAINING_REPO) if f.startswith(f'{CRAFT}/reports/v3/') and f.endswith('.json'))
    load = lambda f: json.load(open(hf_hub_download(TRAINING_REPO, f, token=HF_TOKEN), encoding='utf-8'))
rows = []
for f in files:
    r = load(f)
    pr = r.get('probes')
    if not pr:
        continue
    accs = {k.split('_', 1)[0]: round(v.get('acc', v) if isinstance(v, dict) else v, 2) for k, v in pr.items()}
    led = r.get('ledger') or {}; sp = r.get('special') or {}
    g = r.get('guard') or {}; fired = ','.join(sorted((g.get('fired') or {}).keys())) or '-'
    arms = r.get('arms') or {}
    arm_col = ' '.join(f"{n}:{v.get('fineweb_delta'):+.3f}" for n, v in arms.items() if isinstance(v, dict) and 'fineweb_delta' in v) or '-'
    rows.append((r['tag'][:14], r['step'], round(r['tokens'] / 1e9, 3), led.get('bpb_full'), led.get('toggle_head_aleph_off'),
                 sp.get('doc_bpb'), fired, arm_col, accs))
rows.sort(key=lambda x: x[1])
print(f"{'stage':<15}{'step':>7}{'tok(B)':>8}{'val':>7}{'head':>8}{'doc':>6}  {'guards':<8}{'arms (fineweb delta)':<28}probes")
for tag, step, tk, val, hd, doc, fired, arm_col, accs in rows:
    v = f'{val:.3f}' if isinstance(val, float) else '  --'
    h = f'{hd:+.3f}' if isinstance(hd, float) else '    --'
    d = f'{doc:.1f}' if isinstance(doc, float) else '  --'
    print(f'{tag:<15}{step:>7}{tk:>8}{v:>7}{h:>8}{d:>6}  {fired:<8}{arm_col:<28}{accs}')


In [ ]:
# CARD-ONLY
# C7 — HEAD SCREEN: the instrument for the HEAD_BIRTH decision (S-1004 born-with-function).
# Optional, not part of the routine; runs on a free card at the chosen depth:
# can a live-from-birth head earn contribution in ~1k steps, or does even a
# live head lose to the untrained linear base? Verdict: init-law fix vs v3
# architectural modification. 4 arms x N_STEPS on fresh throwaway crafts;
# arm B/C installs at step 0 = BOUNDARY-WRITE at birth (provenance logged).
import gc, io, json, math, time, torch
import torch.nn.functional as F
from huggingface_hub import HfApi, hf_hub_download
from safetensors.torch import load_file
from geolip.alephllm import get_preset
from geolip.alephllm.model.alephlm import AlephLM
from geolip.alephllm.data.tokenizers import ByteTrigramTokenizer
from geolip.alephllm.data.streams import build_stream
from geolip.alephllm.train.optim import build_optimizers, apply_lr
from geolip.alephllm.train.revival import revive_head
from geolip.alephllm.train.instruments import head_liveness

N_STEPS, GAUGE_EVERY, SEED = 1200, 300, 1337
ARMS = ["A_born_null", "B_revival_birth", "C_matured_transplant",
        "D_live_address_only"]
MATURED = "mini-beatrix-2s/checkpoints/step_00061422.safetensors"  # the 2s MISSION FINAL 16.1B - the matured head (d1024, transplantable)
pcfg = PRESETS[CRAFT]
tok = ByteTrigramTokenizer()
api = HfApi(token=HF_TOKEN)
vs = build_stream("wikitext-103", tok, pcfg.model.context, 4,
                  seed=SEED + 9999, role="val")
VAL = [vs.next_batch().cuda() for _ in range(2)]

def head_toggle(model):
    tot = {True: 0.0, False: 0.0}
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
        for xb in VAL:
            for off in (True, False):
                out = model(xb[:, :-1], targets=xb[:, 1:],
                            disable_head_aleph=off)
                tot[off] += float(out.loss) / len(VAL)
    return (tot[True] - tot[False]) / math.log(2)

def collect_birth(model):
    hs, bl, ys = [], [], []
    hook = model.nf.register_forward_hook(
        lambda m, i, o: hs.append(o.detach().float().reshape(-1, o.shape[-1])))
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
        for xb in VAL:
            out = model(xb[:, :-1], disable_head_aleph=True)
            bl.append(out.logits.float().reshape(-1, 256))
            ys.append(xb[:, 1:].reshape(-1))
    hook.remove()
    return torch.cat(hs), torch.cat(bl), torch.cat(ys)

for arm in ARMS:
    torch.manual_seed(SEED)
    m = AlephLM(pcfg.model).cuda()
    prov = {"arm": arm, "steps": N_STEPS, "seed": SEED, "gauges": []}
    if arm in ("B_revival_birth", "D_live_address_only"):
        m = m.float()
        H, B, Y = collect_birth(m)
        prov["revival"] = revive_head(m, H, B, Y, theta_deg=45.0)
        if arm == "D_live_address_only":
            m.head.w_s.weight.data.zero_()
    if arm == "C_matured_transplant":
        sd = load_file(hf_hub_download(
            "AbstractPhil/alephllm-mini-beatrix-training", MATURED))
        for k in ("head.proj.weight", "head.addr.codebook", "head.w_s.weight"):
            m.state_dict()[k].copy_(sd[k])
    if arm != "A_born_null":                      # frozen address per doctrine
        m.head.proj.weight.requires_grad_(False)
        m.head.addr.codebook.requires_grad_(False)
    m = m.cuda().to(torch.float32)
    opts = build_optimizers(m, pcfg.train.muon_lr, pcfg.train.muon_momentum,
                            pcfg.train.adam_lr)
    st = build_stream("wikitext-103", tok, pcfg.model.context, 8, seed=SEED)
    m.train()
    for step in range(1, N_STEPS + 1):
        apply_lr(opts, [pcfg.train.muon_lr, pcfg.train.adam_lr], step, 200)
        for o in opts:
            o.zero_grad(set_to_none=True)
        xb = st.next_batch().cuda()
        with torch.autocast("cuda", dtype=torch.bfloat16):
            out = m(xb[:, :-1], targets=xb[:, 1:])
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        for o in opts:
            o.step()
        if step % GAUGE_EVERY == 0 or step == N_STEPS:
            m.eval()
            g = dict(step=step,
                     toggle=head_toggle(m),
                     w_s_norm=float(m.head.w_s.weight.norm()),
                     **head_liveness(m, sample_idx=VAL[0][:2, :-1]))
            prov["gauges"].append(g)
            print(f"{arm} step {step}: toggle {g['toggle']:+.4f} "
                  f"live {g['head_liveness_ratio']:.3f}x "
                  f"||W_s|| {g['w_s_norm']:.1f}")
            m.train()
    api.upload_file(
        path_or_fileobj=json.dumps(prov, default=float).encode(),
        path_in_repo=f"{CRAFT}/reports/v3/cultivation_{arm}.json",
        repo_id="AbstractPhil/alephllm-mini-beatrix-training",
        commit_message=f"head cultivation screen: {arm}")
    del m, opts, st
    gc.collect(); torch.cuda.empty_cache()
print("SCREEN COMPLETE - 4 reports under reports/v3/cultivation_*.json")

### Post-training programs (placeholders — the lead's; nothing below is invented)
- **S13 thinking**: trained post-training; think markers are a specials-plane decision (the `THINK` special is reserved in the registry); the honesty gauge is mandatory. `THINK = None`.
- **S6 diversity / condensing terms** (recipe term L): `DIVERSITY = CONDENSING = None`.
- **S7 tool-packaged arms** (recipe term M): `TOOL_ARMS = None`.
- **C3 Level-2 arms (memory accumulators, RelayEMA)**: admitted only after data-route saturation (S11); the head form is packaged (0.8.8 `model/relay.py`), untested under demand (L8 owed).
- **C5 growth clause (A6)**: routed attention from arms adjusted as the model grows — his terms, open.
- **Born-in core-side objectives (D4)**: door CLOSED by L3 (the sub3 wall holds core-side at a catastrophic price, S-1905) — v3 is a steering-and-arms craft.
- **Hub K-alpha gain (A3/A4/S5)**: an inference-parity-certified instrument in the docket, not in the library; training under it is gated on the lead's bed word. `HUB_GAIN = None`.

### Notes
- **Controls**: `<craft>-control` (pure sdpa) is registered beside the craft by C2; any v3 claim runs its twin under the same data plane.
- **Monitoring keys**: `governor/hits_cum` (TB), `guard/<G>` (the step a guard fired), `arms/abstention`, `train/lr_scale` (carries the anneal multiplier); `hub_addr.anchor_merge_pairs` must stay 0.
- **Halts**: a certified guard's archive is `resume/guard_<G>_step<N>.pt`; `resume/latest.pt` is the last healthy checkpoint; the manifest's `halt` field blocks `train()` until a session clears it.
- **Kernels**: eager on Blackwell; `hub_chunk=256`; fp16 forbidden. Sessions are boundary-exact.


In [ ]:
# POST-TRAINING — refuses until a program of record is supplied (see the notes above).
for name in ('THINK', 'DIVERSITY', 'CONDENSING', 'TOOL_ARMS'):
    if globals().get(name) is None:
        print(f'{name}: no program supplied — the lead\'s (plan S13 / S6 / S7); nothing runs')
    else:
        raise SystemExit(f'{name} is set but no post-training routine is built in 0.8.9 — the program is a design item, not a switch')
